# Việc B1 — Lợi thế của GAM có ổn định qua nhiều lần chia dữ liệu không

> **Mentor:** *"Kiểm tra lại lợi thế của GAM trên nhiều lần chia dữ liệu theo thời gian. Nếu GAM
> vẫn tốt hơn ổn định ở chuyến dài và giá cao, các em có thể thử một cách kết hợp GAM–GBM đơn giản."*

## Câu hỏi

Tuần 4 đo lợi thế của GAM ở chuyến dài trên **một** lần chia. Một lần chia không phân biệt được
"lợi thế thật" với "may mắn trên tập test này". Notebook này hỏi: **dấu của chênh lệch có đổi ở lần
chia nào không?**

## Đã có sẵn ba lần chia theo thời gian

Không cần train lại. Pipeline hiện tại vốn train **riêng từng tháng** — mỗi tháng là một lần chia
train/test độc lập theo thời gian. Cộng thêm việc cắt đôi tập test của từng tháng theo thời gian,
ta có **6 lát** để xem độ ổn định.

| Lần chia | Nguồn |
|---|---|
| 2026-01, 2026-02, 2026-03 | Ba tháng, mỗi tháng train/test riêng |
| Nửa đầu / nửa sau mỗi tháng | Cắt tập test theo thời gian |

> ⚠️ Sáu lát này **không độc lập hoàn toàn** — chúng dùng chung bộ sinh dữ liệu và cùng một quy
> trình huấn luyện. Chúng trả lời được câu "lợi thế có ổn định theo thời gian không", không trả lời
> được câu "lợi thế có khái quát sang dữ liệu khác không".

In [ ]:
import warnings, time, sys, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = ("#0072B2", "#E69F00", "#009E73",
                                         "#D55E00", "#CC79A7", "#666666")
INK = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10.5,
})
EVAL = Path("../model/evaluation")
DATA = Path("../data/hcm_train_ready.parquet")
HINH = Path("../docs/hinh_anh"); HINH.mkdir(parents=True, exist_ok=True)
KQ   = Path("ket_qua"); KQ.mkdir(exist_ok=True)

In [ ]:
# ══ NHÓM CỐ ĐỊNH — quy tắc chốt cho cả tuần 5 ══════════════════════════
# Mentor tuần 4: "giữ nguyên nhóm chuyến giữa các model. Không nên để mỗi model
# tự chia nhóm theo giá mà chính nó dự đoán."
# => nhóm chia theo GIÁ THẬT và QUÃNG ĐƯỜNG, hai thứ không phụ thuộc model nào.
CAT_GIA = [0, 50e3, 100e3, 150e3, 200e3, 300e3, np.inf]
TEN_GIA = ["<50k", "50–100k", "100–150k", "150–200k", "200–300k", ">300k"]
CAT_KM  = [0, 2, 5, 8, 12, 15, np.inf]
TEN_KM  = ["<2", "2–5", "5–8", "8–12", "12–15", ">15"]

def gan_nhom(d, cot_gia="y", cot_km="km"):
    d = d.copy()
    d["band"] = pd.cut(d[cot_gia], CAT_GIA, labels=TEN_GIA)
    d["kmb"]  = pd.cut(d[cot_km],  CAT_KM,  labels=TEN_KM)
    return d

def mape(p, y):
    p, y = np.asarray(p, float), np.asarray(y, float)
    return float(np.mean(np.abs(p - y) / y))

def boot_hieu(p_moc, p_moi, y, B=2000, seed=7):
    """CI 95% cho (sai số mốc − sai số mới). Dương = phương án mới TỐT HƠN."""
    y = np.asarray(y, float)
    h = np.abs(np.asarray(p_moc, float) - y)/y - np.abs(np.asarray(p_moi, float) - y)/y
    if len(h) < 2:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    mau = rng.integers(0, len(h), size=(B, len(h)))
    pp = h[mau].mean(axis=1)
    return h.mean(), float(np.percentile(pp, 2.5)), float(np.percentile(pp, 97.5))

def bang_theo_nhom(d, cot_moc, cot_moi, ten_moc="mốc", ten_moi="mới"):
    """MAPE hai phương án trên từng nhóm cố định + CI của chênh lệch."""
    hang = []
    for cot_nhom, nhan in [("kmb", "km"), ("band", "giá thật")]:
        for g, s in d.groupby(cot_nhom, observed=True):
            if len(s) < 30:
                continue
            m, lo, hi = boot_hieu(s[cot_moc], s[cot_moi], s.y)
            hang.append({"Chia theo": nhan, "Nhóm": str(g), "n": len(s),
                         ten_moc: mape(s[cot_moc], s.y), ten_moi: mape(s[cot_moi], s.y),
                         "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                         "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    m, lo, hi = boot_hieu(d[cot_moc], d[cot_moi], d.y)
    hang.append({"Chia theo": "—", "Nhóm": "TOÀN TẬP", "n": len(d),
                 ten_moc: mape(d[cot_moc], d.y), ten_moi: mape(d[cot_moi], d.y),
                 "Chênh (điểm)": m*100, "CI thấp": lo*100, "CI cao": hi*100,
                 "Khác 0": "✔" if (lo > 0 or hi < 0) else ""})
    return pd.DataFrame(hang)

def in_bang(df, cot_mape):
    return df.style.format({**{c: "{:.2%}" for c in cot_mape},
                            "Chênh (điểm)": "{:+.2f}", "CI thấp": "{:+.2f}",
                            "CI cao": "{:+.2f}", "n": "{:,}"}).hide(axis="index")

In [ ]:
# ══ Nạp dự đoán đã có: Hybrid (mốc) và GAM ═════════════════════════════
ut  = pd.read_parquet(EVAL / "uq_pred_test.parquet").reset_index(drop=True)
gam = pd.read_parquet(EVAL / "pred_gam.parquet").reset_index(drop=True)
assert len(ut) == len(gam) and np.allclose(ut.gia_that.values, gam.gia_that.values), \
    "Hai file không cùng thứ tự hàng — không ghép được"

D = pd.DataFrame({
    "thang":  ut.evaluation_month.values,
    "lag":    ut.requested_lag_minutes.values,
    "km":     ut.quote_distance.values,
    "gio":    ut.gio_vn.values,
    "y":      ut.gia_that.values,
    "hybrid": ut.hybrid_pred.values,
    "gam":    gam.hybrid_pred.values,
    "pers":   ut.persistence.values,
})
D = gan_nhom(D[D.lag == 5].reset_index(drop=True))
print(f"Test lag 5 phút: {len(D):,} chuyến · {D.thang.nunique()} tháng")
print(f"MAPE mốc Hybrid {mape(D.hybrid, D.y):.2%} · GAM {mape(D.gam, D.y):.2%}")

## 1. Dựng các lát thời gian

Cắt tập test của mỗi tháng làm đôi theo thứ tự thời gian. Vì dữ liệu trong `uq_pred_test` đã xếp
theo thời gian trong từng tháng, cắt theo chỉ số là cắt theo thời gian.

In [ ]:
ut_t = pd.read_parquet(EVAL / "uq_pred_test.parquet",
                       columns=["evaluation_month", "requested_lag_minutes", "target_timestamp"])
ut_t = ut_t[ut_t.requested_lag_minutes == 5].reset_index(drop=True)
assert len(ut_t) == len(D)
D["ts"] = pd.to_datetime(ut_t.target_timestamp.values, utc=True)
D = D.sort_values(["thang", "ts"]).reset_index(drop=True)

LAT = {}
for th in sorted(D.thang.unique()):
    s = D[D.thang == th]
    LAT[f"{th} · cả tháng"] = s
    giua = len(s) // 2
    LAT[f"{th} · nửa đầu"] = s.iloc[:giua]
    LAT[f"{th} · nửa sau"] = s.iloc[giua:]

for k, v in LAT.items():
    print(f"  {k:22} {len(v):>7,} chuyến   "
          f"{v.ts.min():%d/%m} → {v.ts.max():%d/%m}")

## 2. Chênh lệch GAM − Hybrid trên từng lát, từng nhóm

Dương nghĩa là GAM tốt hơn. Khoảng tin cậy 95% bằng bootstrap 2.000 lần.

In [ ]:
NHOM_XET = [("kmb", ">15"), ("kmb", "12–15"), ("kmb", "8–12"),
            ("band", ">300k"), ("band", "200–300k"), (None, "TOÀN TẬP")]

hang = []
for ten_lat, s in LAT.items():
    for cn, g in NHOM_XET:
        sub = s if cn is None else s[s[cn].astype(str) == g]
        if len(sub) < 30:
            hang.append({"Lát": ten_lat, "Nhóm": g, "n": len(sub),
                         "Hybrid": np.nan, "GAM": np.nan, "Chênh": np.nan,
                         "CI thấp": np.nan, "CI cao": np.nan, "Kết luận": "ít mẫu"})
            continue
        m, lo, hi = boot_hieu(sub.hybrid, sub.gam, sub.y)
        kl = "GAM thắng" if lo > 0 else "Hybrid thắng" if hi < 0 else "không phân biệt"
        hang.append({"Lát": ten_lat, "Nhóm": g, "n": len(sub),
                     "Hybrid": mape(sub.hybrid, sub.y), "GAM": mape(sub.gam, sub.y),
                     "Chênh": m*100, "CI thấp": lo*100, "CI cao": hi*100, "Kết luận": kl})
OD = pd.DataFrame(hang)
OD.style.format({"Hybrid": "{:.2%}", "GAM": "{:.2%}", "Chênh": "{:+.2f}",
                 "CI thấp": "{:+.2f}", "CI cao": "{:+.2f}", "n": "{:,}"}).hide(axis="index")

## 3. Ổn định hay không — nhìn theo từng nhóm

Tiêu chí phải đếm **bằng chứng**, không đếm dấu của ước lượng điểm. Một lát có chênh lệch −0,12
điểm với khoảng tin cậy [−1,75, +1,49] không phải bằng chứng đảo chiều — nó chỉ là nhiễu từ 121
chuyến. Đếm dấu thô sẽ kết luận "đổi dấu" và làm hỏng cả hướng đi.

Quy tắc dùng ở đây: chỉ tính là bằng chứng khi **khoảng tin cậy loại trừ được 0**.

| Kết luận | Điều kiện |
|---|---|
| ✔ ổn định · GAM thắng | có lát GAM thắng rõ, **không** lát nào Hybrid thắng rõ |
| ✔ ổn định · Hybrid thắng | ngược lại |
| ✘ mâu thuẫn | có cả lát GAM thắng rõ lẫn lát Hybrid thắng rõ |
| — không đủ bằng chứng | không lát nào loại trừ được 0 |

In [ ]:
tt = []
for g in [x[1] for x in NHOM_XET]:
    s = OD[(OD.Nhóm == g) & OD.Chênh.notna()]
    if not len(s):
        continue
    thang = int((s["CI thấp"] > 0).sum())        # lat GAM thang RO
    thua  = int((s["CI cao"] < 0).sum())         # lat Hybrid thang RO
    if thang and not thua:
        kl = "✔ ổn định · GAM thắng"
    elif thua and not thang:
        kl = "✔ ổn định · Hybrid thắng"
    elif thang and thua:
        kl = "✘ mâu thuẫn"
    else:
        kl = "— không đủ bằng chứng"
    tt.append({"Nhóm": g, "Số lát": len(s),
               "Lát GAM thắng rõ": thang, "Lát Hybrid thắng rõ": thua,
               "Lát không phân biệt": len(s) - thang - thua,
               "Chênh nhỏ nhất": s.Chênh.min(), "Chênh lớn nhất": s.Chênh.max(),
               "Kết luận": kl})
TT = pd.DataFrame(tt)
TT.style.format({"Chênh nhỏ nhất": "{:+.2f}", "Chênh lớn nhất": "{:+.2f}"}).hide(axis="index")

In [ ]:
# ═════════ HÌNH OD1 — forest plot: moi nhom mot cum lat ═════════
ve = [g for g in [">15", "12–15", ">300k", "TOÀN TẬP"]
      if OD[(OD.Nhóm == g) & OD.Chênh.notna()].shape[0] > 0]
fig, ax = plt.subplots(1, len(ve), figsize=(4.1*len(ve), 5), sharex=False)
ax = np.atleast_1d(ax)

for a, g in zip(ax, ve):
    s = OD[(OD.Nhóm == g) & OD.Chênh.notna()].reset_index(drop=True)
    yy = np.arange(len(s))[::-1]
    for i, r in s.iterrows():
        y = yy[i]
        ro = (r["CI thấp"] > 0) or (r["CI cao"] < 0)
        mau = GREEN if (ro and r.Chênh > 0) else RED if ro else MUT
        ca_thang = "cả tháng" in r["Lát"]
        a.plot([r["CI thấp"], r["CI cao"]], [y, y], color=mau,
               lw=3.0 if ca_thang else 1.8, alpha=.9 if ca_thang else .6)
        a.plot(r.Chênh, y, "o", color=mau, ms=9 if ca_thang else 6)
    a.axvline(0, color=INK, lw=1.2)
    a.set_yticks(yy)
    a.set_yticklabels([r["Lát"].replace("2026-", "") for _, r in s.iterrows()], fontsize=9)
    a.set_xlabel("GAM − Hybrid (điểm)")
    on = TT[TT.Nhóm == g]
    a.set_title(f"{g}\n{on['Kết luận'].iloc[0] if len(on) else ''}",
                fontweight="bold", fontsize=11)

fig.suptitle("OD1 — Lợi thế của GAM qua 9 lát thời gian, khoảng tin cậy 95%\n"
             "Bên phải vạch 0 = GAM tốt hơn · nét đậm = cả tháng · xám = không phân biệt được",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
fig.savefig(HINH / "OD1_gam_on_dinh.png")
plt.show()

## 4. Lợi thế thay đổi thế nào theo quãng đường

Nếu định ghép GAM–GBM theo quãng đường ở việc B2 thì cần biết **điểm giao** — từ mốc km nào GAM bắt
đầu thắng.

In [ ]:
BIEN = [0, 2, 4, 6, 8, 10, 12, 14, 16, 20, np.inf]
D["km_min"] = pd.cut(D.km, BIEN)
r = []
for g, s in D.groupby("km_min", observed=True):
    if len(s) < 100:
        continue
    m, lo, hi = boot_hieu(s.hybrid, s.gam, s.y, B=1000)
    r.append(dict(giua=(g.left + min(g.right, 22))/2, n=len(s),
                  chenh=m*100, lo=lo*100, hi=hi*100))
GD = pd.DataFrame(r)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(GD.giua, GD.lo, GD.hi, color=GREEN, alpha=.16, label="CI 95%")
ax.plot(GD.giua, GD.chenh, "o-", color=GREEN, lw=2.4, ms=7, label="GAM − Hybrid")
ax.axhline(0, color=INK, lw=1.3)
giao = GD[GD.chenh > 0]
if len(giao):
    ax.axvline(giao.giua.min(), color=ORANGE, ls="--", lw=1.6,
               label=f"GAM bắt đầu thắng ~{giao.giua.min():.0f} km")
for _, q in GD.iterrows():
    ax.annotate(f"{q.n:,.0f}", (q.giua, q.hi), fontsize=8, color=MUT,
                ha="center", xytext=(0, 4), textcoords="offset points")
ax.set_xlabel("Quãng đường (km)")
ax.set_ylabel("Chênh lệch MAPE (điểm) — dương = GAM tốt hơn")
ax.set_title("OD2 — Lợi thế của GAM tăng dần theo quãng đường\n"
             "Số phía trên là số chuyến trong mỗi khoảng",
             fontweight="bold", fontsize=12.5)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(HINH / "OD2_loi_the_theo_km.png")
plt.show()

## 5. Lưu kết quả

In [ ]:
OD.to_csv(KQ / "B1_on_dinh_theo_lat.csv", index=False)
TT.to_csv(KQ / "B1_tong_ket_on_dinh.csv", index=False)
GD.to_csv(KQ / "B1_loi_the_theo_km.csv", index=False)

on_dinh_km = TT[TT.Nhóm == ">15"]["Kết luận"].iloc[0] if (TT.Nhóm == ">15").any() else "?"
giao_km = float(GD[GD.chenh > 0].giua.min()) if (GD.chenh > 0).any() else None
json.dump({"on_dinh_>15km": on_dinh_km, "diem_giao_km": giao_km},
          open(KQ / "B1_ket_luan.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

print("Đã lưu vào", KQ.resolve())
print(f"\nNhóm >15 km: {on_dinh_km}")
print(f"GAM bắt đầu thắng từ khoảng: {giao_km} km"
      if giao_km else "GAM không thắng ở khoảng km nào")
if on_dinh_km.startswith("✔ ổn định · GAM"):
    print("\n→ ĐẠT cổng: chạy tiếp 03_GHEP_GAM_GBM.ipynb")
elif on_dinh_km.startswith("✘"):
    print("\n→ KHÔNG đạt: các lát mâu thuẫn nhau. Dừng, báo cáo kết quả âm tính.")
else:
    print("\n→ Chưa đủ bằng chứng: nhóm quá thưa. Cân nhắc gộp nhóm hoặc bỏ hướng ghép.")

## 6. Kết luận cần điền

1. Nhóm `>15 km`: có lát nào Hybrid thắng **rõ** (CI loại trừ 0) không?
2. Nhóm `>300k`: có ổn định không, hay chỉ thắng ở một vài tháng?
3. Điểm giao theo quãng đường nằm ở đâu — con số này là đầu vào cho `d₀` ở việc B2.

**Cổng quyết định:** chỉ chạy tiếp `03_GHEP_GAM_GBM` nếu nhóm `>15 km` ra
"✔ ổn định · GAM thắng". Ghép hai model dựa
trên một lợi thế không ổn định là làm phức tạp hệ thống mà không được gì.